# Langfuse Multi-Agent Observability Demo
Customer Support System with Router, Billing, Technical, and Response Agents

In [ ]:
!pip install langfuse requests

In [1]:
from langfuse import Langfuse
import requests

LANGFUSE_PUBLIC_KEY = "your-public-key"
LANGFUSE_SECRET_KEY = "your-secret-key"

langfuse = Langfuse(
    public_key=LANGFUSE_PUBLIC_KEY,
    secret_key=LANGFUSE_SECRET_KEY,
    host="https://api.langfuse.com"
)

In [6]:
# ============================================================
# 1. INSTALL
# ============================================================

%pip install -U langfuse requests


# ============================================================
# 2. IMPORTS
# ============================================================

import requests

from langfuse import get_client

# Uses:
# LANGFUSE_PUBLIC_KEY
# LANGFUSE_SECRET_KEY
# LANGFUSE_HOST
#
# from environment variables

langfuse = get_client()

print("Langfuse initialized")


# ============================================================
# 3. HELPER FUNCTION
# ============================================================

def fetch_customer_data(user_id: int):

    response = requests.get(
        f"https://jsonplaceholder.typicode.com/users/{user_id}",
        timeout=10,
    )

    response.raise_for_status()

    return response.json()


# ============================================================
# 4. ROUTER AGENT
# ============================================================

def router_agent(query: str):

    with langfuse.start_as_current_observation(
        name="router_agent",
        as_type="span",
        input={"query": query},
    ) as span:

        if "bill" in query.lower():
            intent = "billing"

        elif (
            "error" in query.lower()
            or "issue" in query.lower()
            or "problem" in query.lower()
        ):
            intent = "technical"

        else:
            intent = "general"

        span.update(
            output={"intent": intent}
        )

        return intent


# ============================================================
# 5. BILLING AGENT
# ============================================================

def billing_agent(user_id: int):

    with langfuse.start_as_current_observation(
        name="billing_agent",
        as_type="span",
        input={"user_id": user_id},
    ) as span:

        customer = fetch_customer_data(user_id)

        bill_amount = len(customer["name"]) * 10

        result = {
            "customer_name": customer["name"],
            "bill_amount": bill_amount,
        }

        span.update(
            output=result
        )

        return result


# ============================================================
# 6. TECHNICAL AGENT
# ============================================================

def technical_agent(query: str):

    with langfuse.start_as_current_observation(
        name="technical_agent",
        as_type="span",
        input={"query": query},
    ) as span:

        diagnosis = (
            "Reset the application, clear cache, "
            "and try again."
        )

        span.update(
            output={
                "diagnosis": diagnosis
            }
        )

        return diagnosis


# ============================================================
# 7. RESPONSE AGENT
# ============================================================

def response_agent(context):

    with langfuse.start_as_current_observation(
        name="response_agent",
        as_type="generation",
        model="support-response-generator",
    ) as generation:

        if isinstance(context, dict):

            response = (
                f"Customer {context['customer_name']} "
                f"has a bill of "
                f"${context['bill_amount']}."
            )

        else:

            response = (
                f"Recommended solution: {context}"
            )

        generation.update(
            output=response
        )

        return response


# ============================================================
# 8. MAIN PIPELINE
# ============================================================

def run_pipeline(
    query: str,
    user_id: int = 1,
):

    with langfuse.start_as_current_observation(
        name="customer_support_pipeline",
        as_type="span",
        input={
            "query": query,
            "user_id": user_id,
        },
    ) as root:

        # -----------------------------
        # Router
        # -----------------------------
        intent = router_agent(query)

        # -----------------------------
        # Branch
        # -----------------------------
        if intent == "billing":

            result = billing_agent(user_id)

        elif intent == "technical":

            result = technical_agent(query)

        else:

            result = (
                "General support will contact you."
            )

        # -----------------------------
        # Final response
        # -----------------------------
        response = response_agent(result)

        root.update(
            output={
                "intent": intent,
                "result": result,
                "response": response,
            }
        )

        return response


# ============================================================
# 9. TEST
# ============================================================

print(
    run_pipeline(
        "I have a billing issue",
        user_id=1,
    )
)

print()

print(
    run_pipeline(
        "My application has an error",
        user_id=1,
    )
)

print()

print(
    run_pipeline(
        "Need help with my account",
        user_id=1,
    )
)


# ============================================================
# 10. FLUSH EVENTS
# ============================================================

langfuse.flush()

print("All traces sent to Langfuse.")

Note: you may need to restart the kernel to use updated packages.
Langfuse initialized
Customer Leanne Graham has a bill of $130.

Recommended solution: Reset the application, clear cache, and try again.

Recommended solution: General support will contact you.
All traces sent to Langfuse.


In [13]:
# %pip install --upgrade langfuse

# %pip uninstall -y langfuse
# %pip uninstall -y langfuse   # run twice to be sure
# %pip install --upgrade langfuse
%pip show langfuse



Name: langfuse
Version: 4.7.1
Summary: A client library for accessing langfuse
Home-page: 
Author: langfuse
Author-email: langfuse <developers@langfuse.com>
License-Expression: MIT
Location: d:\Trainings\AI\abu dhabhi agent\.venv\Lib\site-packages
Requires: backoff, httpx, opentelemetry-api, opentelemetry-exporter-otlp-proto-http, opentelemetry-sdk, packaging, pydantic, wrapt
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [8]:
print(run_pipeline("I have a billing issue"))
print(run_pipeline("My app has an error"))

Customer Leanne Graham has a bill of $130.
Recommended solution: Reset the application, clear cache, and try again.


In [ ]:
# ============================================================
# LANGFUSE 4.7.1 - FULL OBSERVABILITY PIPELINE (CODE ONLY)
# ============================================================

%pip install -U langfuse requests

import time
import requests
from langfuse import get_client

langfuse = get_client()


# ============================================================
# MOCK LLM WITH USAGE + LATENCY
# ============================================================

def call_llm(prompt: str, model: str = "gpt-4o-mini"):

    start = time.time()

    with langfuse.start_as_current_observation(
        name="llm_call",
        as_type="generation",
        model=model,
        input={"prompt": prompt}
    ) as obs:

        response = f"[{model}] {prompt[:50]}..."

        usage = {
            "input_tokens": len(prompt.split()),
            "output_tokens": len(response.split()),
            "total_tokens": len(prompt.split()) + len(response.split())
        }

        obs.update(
            output={"text": response},
            usage=usage,
            metadata={"latency_sec": time.time() - start}
        )

        return response


# ============================================================
# TOOL: FETCH CUSTOMER DATA
# ============================================================

def fetch_customer_data(user_id: int):

    with langfuse.start_as_current_observation(
        name="fetch_customer_data",
        as_type="tool",
        input={"user_id": user_id}
    ) as obs:

        data = requests.get(
            f"https://jsonplaceholder.typicode.com/users/{user_id}"
        ).json()

        obs.update(output=data)

        return data


# ============================================================
# ROUTER AGENT
# ============================================================

def router_agent(query: str):

    with langfuse.start_as_current_observation(
        name="router_agent",
        as_type="agent",
        input={"query": query}
    ) as obs:

        llm_result = call_llm(f"Classify intent: {query}")

        if "bill" in query.lower():
            intent = "billing"
        elif "error" in query.lower():
            intent = "technical"
        else:
            intent = "general"

        obs.update(
            output={
                "intent": intent,
                "llm": llm_result
            }
        )

        return intent


# ============================================================
# BILLING AGENT
# ============================================================

def billing_agent(user_id: int):

    with langfuse.start_as_current_observation(
        name="billing_agent",
        as_type="tool",
        input={"user_id": user_id}
    ) as obs:

        data = fetch_customer_data(user_id)

        result = {
            "user": data["name"],
            "bill": len(data["name"]) * 12
        }

        obs.update(output=result)

        return result


# ============================================================
# TECHNICAL AGENT
# ============================================================

def technical_agent(query: str):

    with langfuse.start_as_current_observation(
        name="technical_agent",
        as_type="agent",
        input={"query": query}
    ) as obs:

        diagnosis = call_llm(f"Troubleshoot: {query}")

        obs.update(output={"diagnosis": diagnosis})

        return diagnosis


# ============================================================
# RESPONSE AGENT
# ============================================================

def response_agent(context):

    with langfuse.start_as_current_observation(
        name="response_agent",
        as_type="generation",
        model="response-model"
    ) as obs:

        prompt = f"Create final response from: {context}"

        response = call_llm(prompt)

        obs.update(output={"response": response})

        return response


# ============================================================
# PIPELINE
# ============================================================

def run_pipeline(query: str, user_id: int = 1):

    with langfuse.start_as_current_observation(
        name="customer_support_pipeline",
        as_type="agent",
        input={"query": query}
    ) as root:

        intent = router_agent(query)

        if intent == "billing":
            result = billing_agent(user_id)

        elif intent == "technical":
            result = technical_agent(query)

        else:
            result = "general support"

        response = response_agent(result)

        root.update(
            output={
                "intent": intent,
                "response": response
            }
        )

        return response


# ============================================================
# TEST
# ============================================================

print(run_pipeline("I have a billing issue", 1))
print(run_pipeline("My app has error", 2))
print(run_pipeline("Need help", 3))


# ============================================================
# FLUSH
# ============================================================

langfuse.flush()

In [19]:
# ============================================================
# INSTALL
# ============================================================

%pip install -U langfuse requests

import uuid
import time
import requests
from langfuse import get_client

langfuse = get_client()


# ============================================================
# MOCK AGENTS
# ============================================================

def router_agent(prompt):
    if "bill" in prompt.lower():
        return "billing"
    elif "error" in prompt.lower():
        return "technical"
    else:
        return "general"


def billing_agent(user_id):
    return {"user_id": user_id, "bill": 120}


def technical_agent(prompt):
    return "Restart app and clear cache"


def response_agent(result):
    return f"Final response: {result}"


# ============================================================
# PIPELINE WITH SESSION (CORRECT WAY)
# ============================================================

def run_chat(user_prompt: str, user_id: str):

    session_id = str(uuid.uuid4())

    # ROOT TRACE (THIS IS YOUR SESSION)
    with langfuse.start_as_current_observation(
        name="chat_session",
        as_type="agent",
        input={
            "user_prompt": user_prompt,
            "session_id": session_id,
            "user_id": user_id
        },
        metadata={
            "session_id": session_id,
            "user_id": user_id
        }
    ) as session:

        # ---------------- ROUTER ----------------
        with langfuse.start_as_current_observation(
            name="router",
            as_type="agent",
            input={"prompt": user_prompt}
        ) as router:

            intent = router_agent(user_prompt)
            router.update(output={"intent": intent})

        # ---------------- BRANCH ----------------
        if intent == "billing":

            with langfuse.start_as_current_observation(
                name="billing_agent",
                as_type="tool",
                input={"user_id": user_id}
            ) as span:

                result = billing_agent(user_id)
                span.update(output=result)

        elif intent == "technical":

            with langfuse.start_as_current_observation(
                name="technical_agent",
                as_type="tool",
                input={"query": user_prompt}
            ) as span:

                result = technical_agent(user_prompt)
                span.update(output={"result": result})

        else:
            result = "General support flow"

        # ---------------- RESPONSE (LLM STYLE) ----------------
        with langfuse.start_as_current_observation(
            name="response_agent",
            as_type="generation",
            model="support-model-v1",
            input={"context": result}
        ) as span:

            response = response_agent(result)
            span.update(output={"response": response})

        # ROOT OUTPUT
        session.update(
            output={
                "session_id": session_id,
                "intent": intent,
                "response": response
            }
        )

    return session_id, response


# ============================================================
# TEST CASES
# ============================================================

print("\n===== TEST 1 =====")
sid1, res1 = run_chat("I have a billing issue", "user_1")
print(sid1, res1)

print("\n===== TEST 2 =====")
sid2, res2 = run_chat("My app has error", "user_2")
print(sid2, res2)

print("\n===== TEST 3 =====")
sid3, res3 = run_chat("Need help with account", "user_3")
print(sid3, res3)


# ============================================================
# FLUSH
# ============================================================

langfuse.flush()

print("DONE - traces sent to Langfuse")

Note: you may need to restart the kernel to use updated packages.

===== TEST 1 =====
bf90a287-150c-45a2-ba7c-7bfa997ceeb6 Final response: {'user_id': 'user_1', 'bill': 120}

===== TEST 2 =====
e8f7d3e8-79e8-4597-8961-1c3959d39ccb Final response: Restart app and clear cache

===== TEST 3 =====
acd63cfe-362b-444f-a81d-f190cde77c32 Final response: General support flow
DONE - traces sent to Langfuse


In [25]:
# ============================================================
# FIXED: OPENROUTER (QWEN) + LANGFUSE 4.7.1 PIPELINE
# ============================================================

#%pip install -U langfuse openai requests

import os
import uuid
import time
import requests
from openai import OpenAI
from langfuse import get_client


# ============================================================
# CONFIG
# ============================================================

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

MODEL = "qwen/qwen-2.5-72b-instruct"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    default_headers={
        "HTTP-Referer": "http://localhost",
        "X-Title": "Langfuse-Qwen-Demo"
    }
)

langfuse = get_client()


# ============================================================
# LLM CALL (QWEN + COST + LANGFUSE)
# ============================================================

def call_llm(prompt: str, model=MODEL):

    start = time.time()

    with langfuse.start_as_current_observation(
        name="llm_call",
        as_type="generation",
        model=model,
        input={"prompt": prompt}
    ) as obs:

        resp = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": "You are a helpful assistant."},
                {"role": "user", "content": prompt}
            ],
            max_tokens=100,
        )

        text = resp.choices[0].message.content
        usage = resp.usage

        obs.update(
            output={"text": text},
            usage={
                "input_tokens": usage.prompt_tokens,
                "output_tokens": usage.completion_tokens,
                "total_tokens": usage.total_tokens
            },
            metadata={
                "latency_sec": time.time() - start
            }
        )

        return text


# ============================================================
# TOOL
# ============================================================

def fetch_customer_data(user_id: int, session_id: str):

    with langfuse.start_as_current_observation(
        name="fetch_customer_data",
        as_type="tool",
        input={"user_id": user_id},
        metadata={"session_id": session_id}
    ) as obs:

        data = requests.get(
            f"https://jsonplaceholder.typicode.com/users/{user_id}"
        ).json()

        obs.update(output=data)

        return data


# ============================================================
# ROUTER
# ============================================================

def router_agent(query: str, session_id: str):

    with langfuse.start_as_current_observation(
        name="router_agent",
        as_type="agent",
        input={"query": query},
        metadata={"session_id": session_id}
    ) as obs:

        llm_out = call_llm(f"Classify intent: {query}")

        if "bill" in query.lower():
            intent = "billing"
        elif "error" in query.lower():
            intent = "technical"
        else:
            intent = "general"

        obs.update(
            output={
                "intent": intent,
                "llm": llm_out
            }
        )

        return intent


# ============================================================
# BILLING
# ============================================================

def billing_agent(user_id: int, session_id: str):

    with langfuse.start_as_current_observation(
        name="billing_agent",
        as_type="tool",
        input={"user_id": user_id},
        metadata={"session_id": session_id}
    ) as obs:

        data = fetch_customer_data(user_id, session_id)

        result = {
            "user": data["name"],
            "bill": len(data["name"]) * 12
        }

        obs.update(output=result)

        return result


# ============================================================
# TECHNICAL
# ============================================================

def technical_agent(query: str, session_id: str):

    with langfuse.start_as_current_observation(
        name="technical_agent",
        as_type="agent",
        input={"query": query},
        metadata={"session_id": session_id}
    ) as obs:

        result = call_llm(f"Troubleshoot this issue: {query}")

        obs.update(output={"diagnosis": result})

        return result


# ============================================================
# RESPONSE
# ============================================================

def response_agent(context, session_id: str):

    with langfuse.start_as_current_observation(
        name="response_agent",
        as_type="generation",
        model=MODEL,
        metadata={"session_id": session_id}
    ) as obs:

        result = call_llm(f"Create support response: {context}")

        obs.update(output={"response": result})

        return result


# ============================================================
# PIPELINE (SESSION + QWEN + LANGFUSE)
# ============================================================

def run_pipeline(query: str, user_id: int = 1):

    session_id = str(uuid.uuid4())

    with langfuse.start_as_current_observation(
        name="customer_support_session",
        as_type="agent",
        input={"query": query},
        metadata={
            "session_id": session_id,
            "user_id": user_id
        }
    ) as root:

        intent = router_agent(query, session_id)

        if intent == "billing":
            result = billing_agent(user_id, session_id)

        elif intent == "technical":
            result = technical_agent(query, session_id)

        else:
            result = "general support"

        response = response_agent(result, session_id)

        root.update(
            output={
                "intent": intent,
                "response": response
            }
        )

        return session_id, response


# ============================================================
# TESTS
# ============================================================

sid1, r1 = run_pipeline("I have a billing issue", 1)
print(sid1, r1)

sid2, r2 = run_pipeline("My app has error", 2)
print(sid2, r2)

sid3, r3 = run_pipeline("Need help", 3)
print(sid3, r3)


# ============================================================
# FLUSH
# ============================================================

langfuse.flush()

f0799056-8fc5-4d9d-8982-8400141fadab Hi Leanne,

Thank you for reaching out to us. I see that your recent bill amount is $156. If you have any questions or concerns about this bill, I'm here to help!

Here are a few options to assist you:

1. **Review Your Bill**: You can log into your account to review the details of your recent charges. This will help you understand where each charge came from.
2. **Payment Options**: If you need assistance with making a payment, we
7e91ce96-701b-4442-8048-88a9c2243b86 Certainly! To help troubleshoot the issue with your app, I'll need a bit more information. Here are some questions to guide us through the process:

1. **What is the error message?**
   - Please provide the exact error message you are seeing. This can often give us a clue about what's going wrong.

2. **What platform is your app running on?**
   - Is it a web app, a mobile app (iOS or Android), or a desktop application?

3
195bcb76-e5a4-4352-bff1-edd2df0de997 Certainly! Below is a temp